# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and basic processing of the FAIR² dataset package using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
- [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

The dataset's Croissant schema provides full metadata, record sets, and field definitions, enabling programmatic and standards-based access.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and get the dataset object
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}\n")
print("Description:")
print(dataset.metadata.description)


## 2. Data Overview
Review the available record sets, fields, their names, and corresponding `@id`s.
We use the Croissant dataset interface to enumerate all record sets, each with a unique `@id`.


In [ ]:
print("Record sets in dataset:\n")
record_set_objs = dataset.record_sets()
rs_ids = []

for rs in record_set_objs:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '-')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | dtype: {getattr(field, 'data_type', '-')}")
    print()
    rs_ids.append(rs.id)


## 3. Data Extraction
Load all records from each record set into a dictionary of pandas DataFrames using their `@id`.
You can use these DataFrames for further analysis, referencing fields by their `@id`.

In [ ]:
# Collect all DataFrames for the available record sets
dataframes = {}

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))


### Example: Select the main tabular record set
For this dataset, we select the main record set containing the patient and clinicopathological variables. Below, update `main_record_set_id` if a different one should be chosen (see previous cell output for available record set `@id`s).

In [ ]:
# Identify the main record set for analysis (update the value as needed)
if rs_ids:
    main_record_set_id = rs_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"Main record set @id: {main_record_set_id}")
    print(f"Available columns:")
    print(main_df.columns.tolist())
    display(main_df.head(10))
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply key exploratory analysis and filtering using field `@id`.
Below, select one numeric field and one grouping field (see DataFrame columns above and Croissant metadata for `@id`s).

In [ ]:
# Example: Choose a numeric field and a grouping field by their @id (update as needed)
all_columns = main_df.columns.tolist()

# Find a likely numeric field by looking for 'age', 'interval', 'count', etc in column names
numeric_field_id = next((col for col in all_columns if 'age' in col.lower()), all_columns[0])
print(f"Selected numeric field (@id): {numeric_field_id}")

group_field_id = next((col for col in all_columns if ('sex' in col.lower() or 'gender' in col.lower())), 
                    (all_columns[1] if len(all_columns) > 1 else all_columns[0]))
print(f"Selected group field  (@id): {group_field_id}\n")

# Filter and normalize
if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    threshold = main_df[numeric_field_id].mean()
else:
    # Try coercing to numeric
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean()

filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.1f}:")
display(filtered_df[[numeric_field_id, group_field_id]].head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)

print(f"Normalized {numeric_field_id} values:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping/aggregation
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df)


## 5. Visualization
Visualize the distribution of the numeric field and group averages.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
fig, ax = plt.subplots(1,2, figsize=(12, 4))
sns.histplot(main_df[numeric_field_id], kde=True, ax=ax[0])
ax[0].set_title(f"Histogram of {numeric_field_id}")
ax[0].set_xlabel(numeric_field_id)
ax[0].set_ylabel("Frequency")

# Boxplot by group
if group_field_id in main_df.columns:
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df, ax=ax[1])
    ax[1].set_title(f"{numeric_field_id} by {group_field_id}")

plt.tight_layout()
plt.show()

## 6. Conclusion
This notebook demonstrated how to programmatically access and process a FAIR²-compliant clinical dataset using `mlcroissant`.

- Metadata, record sets, and fields are always referenced by their Croissant `@id`.
- Data was dynamically loaded and explored using variables and Croissant dataset primitives.
- We performed basic EDA: filtered by a numeric field, normalized it, and grouped by a selected category.
- Data visualizations provided an initial look into distributions and possible grouping effects.

**You can extend this template for advanced analysis, feature engineering, or ML modeling while maintaining provenance and reference integrity via Croissant `@id`s!**